# 2. Random Variables — Characterizing the System's Inputs

**Building a Heart Disease Risk-Screening System — Notebook 2 of 12, Stage 1: Understanding the Raw Signals**

Every column in the registry — age, cholesterol, resting blood pressure, whether a
patient has disease — is a **random variable**: a quantity that varies from patient
to patient in a way we can describe statistically, even though we can't predict any
one patient's exact value in advance. This notebook builds the vocabulary
(expectation, variance, standardization) needed to describe each input precisely,
before Notebook 3 asks what *shape* each one's variation actually takes.

## The topic

A random variable is **discrete** (a countable set of values — `sex`, `target`,
`cp`) or **continuous** (any value in a range — `age`, `chol`, `trestbps`). Every
one of them has an expected value (its central tendency) and a variance (how much
it typically deviates from that center) — the two numbers that summarize *how a
variable behaves* before you ever look at its relationship to anything else.

## Why it matters for this system

A predictive model is, mechanically, a function that combines many random
variables into one output. If you don't know how *volatile* an input is
individually, you can't judge whether the model is behaving sensibly with it —
and inputs measured on wildly different scales (age in years vs. cholesterol in
mg/dL) need to be put on comparable footing before they can be combined fairly,
which is exactly what standardization (covered below) is for.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(df.shape)
df[["age", "chol", "trestbps", "thalach", "target"]].describe().round(1)

## The toolkit

| Tool | Describes |
|---|---|
| **Expected value (mean)** | Where the variable centers |
| **Variance / standard deviation** | How much it typically spreads around that center |
| **`ddof` (Bessel's correction)** | Whether you're describing a known population or estimating from a sample |
| **Linearity of expectation** | Combining variables' means without needing independence |
| **z-score (standardization)** | Putting differently-scaled variables on one common scale |
| **Skewness** | Whether the variable's spread is lopsided |

## How to choose

Use mean/variance any time you need "typical value" and "how much it varies" —
they're the default first move on any new column. Use `ddof=1` (the default) when
your data is a *sample* standing in for a larger population — true here, since this
registry is one clinic's patients, not every patient who could exist. Reach for
z-scores specifically when comparing or combining variables measured in different
units — age and cholesterol are not directly comparable numbers until standardized.
Check skewness before assuming a variable is roughly symmetric, since several tools
later in this module (and the correlation/regression stages) work best on
roughly-symmetric inputs.

## Applied to the registry

### Expected value and variance, on two different scales

In [ ]:
print("chol -- a continuous input")
print(f"  mean:   {df['chol'].mean():.1f} mg/dL")
print(f"  std:    {df['chol'].std():.1f} mg/dL")

print("\ntarget -- a discrete (Bernoulli) input")
print(f"  mean:   {df['target'].mean():.3f}   (for a 0/1 variable, the mean IS the probability of a 1)")

### `ddof`: sample vs. population variance

In [ ]:
manual_var = ((df["chol"] - df["chol"].mean()) ** 2).mean()
print(f"Manual variance (divide by n):        {manual_var:.2f}")
print(f"pandas var(ddof=0) -- population:     {df['chol'].var(ddof=0):.2f}")
print(f"pandas var(ddof=1) -- sample default: {df['chol'].var(ddof=1):.2f}")
print("\nThis registry is a SAMPLE of the clinic's patient population (more patients will")
print("be added over time), so ddof=1 -- the default -- is the correct choice throughout")
print("this module.")

### Conditional expectation: the same idea as Notebook 1, applied to a mean instead of a rate

Just as conditioning shifted a probability in Notebook 1, it shifts an expected
value here — the average cholesterol *within* the disease group versus outside it
is a first, simple look at whether cholesterol distinguishes the two groups (a
question Notebook 7's t-test will answer rigorously).

In [ ]:
chol_by_group = df.groupby("target")["chol"].agg(["mean", "std", "count"])
chol_by_group.index = ["no disease", "disease"]
print(chol_by_group.round(1))

### Linearity of expectation: combining variables without needing independence

$$E[aX + b] = a\,E[X] + b$$

This holds regardless of correlation between variables — useful the moment the
system needs to combine several inputs into a derived score.

In [ ]:
# A simple derived score: 2x cholesterol contribution + 1x resting blood pressure contribution
E_chol, E_bp = df["chol"].mean(), df["trestbps"].mean()
E_derived_by_linearity = 2 * E_chol + 1 * E_bp

df["derived_score"] = 2 * df["chol"] + df["trestbps"]
E_derived_actual = df["derived_score"].mean()

print(f"E[2*chol + trestbps] via linearity: {E_derived_by_linearity:.1f}")
print(f"Actual mean of the combined column:  {E_derived_actual:.1f}   (matches, as it must)")

### Standardization: putting every input on the same footing

`age` (years) and `chol` (mg/dL) are not directly comparable — a z-score
re-expresses each in units of "standard deviations from its own mean," which is
also literally what feeding these into most models expects.

In [ ]:
df["age_z"] = (df["age"] - df["age"].mean()) / df["age"].std()
df["chol_z"] = (df["chol"] - df["chol"].mean()) / df["chol"].std()

most_unusual_age = df.loc[df["age_z"].abs().idxmax()]
most_unusual_chol = df.loc[df["chol_z"].abs().idxmax()]

print(f"Most unusual age:  {most_unusual_age['age']:.0f} years  (z={most_unusual_age['age_z']:.2f})")
print(f"Most unusual chol: {most_unusual_chol['chol']:.0f} mg/dL (z={most_unusual_chol['chol_z']:.2f})")
print("\nBoth 'unusual' scores are now on the same -3-to-+3-ish scale, directly comparable")
print("even though the raw units aren't.")

### Skewness: is this input symmetric, or lopsided?

In [ ]:
from scipy.stats import skew

for col in ["age", "chol", "trestbps", "thalach"]:
    print(f"{col:10} skewness = {skew(df[col]):+.2f}")
print("\nValues near 0 are roughly symmetric; positive values have a long right tail")
print("(a few unusually high measurements pulling the mean above the typical value) --")
print("Notebook 3 picks this up directly when deciding which distribution fits each input.")

## Systems view — what this stage hands to the next one

Every input the eventual model will use now has a documented center, spread, and
symmetry. That's the raw material Notebook 3 needs to ask a sharper question: not
just "where does this variable center and how much does it spread," but "what
*shape* does its variation take" — which determines which statistical tools are
even valid to apply to it later in the pipeline.

## Try it yourself

1. Compute `E[thalach | target==1]` and `E[thalach | target==0]` the same way
   `chol` was split above — does max heart rate separate the two groups more or
   less than cholesterol did?
2. Using linearity of expectation, predict what `E[3*trestbps - chol]` should be
   without computing the combined column directly, then verify.
3. Find the input with the highest skewness among `age`, `chol`, `trestbps`,
   `thalach` — that's the notebook 3 will need to work hardest to model correctly.